# Apply Chirp Waveforms To The NVNMR Sensing Sequence

Load the NVNMR adiabatic-inversion-train TPS/TNT pair, write up/down chirp phase tables into `phmod_p180` and `phmod_pn180`, synchronize frequency and timing parameters, and save the output experiment through TNMR.


## 1. Parameters

Edit these values for a different chirp. `freq_neg` and `freq_pos` are derived from half of `SWEEP_WIDTH_HZ`.


In [ ]:
POINTS = 20000
DWELL_US = 0.2
SWEEP_WIDTH_HZ = 200.0e3

RUN_TNMR_SAVE_AS = True
TNMR_LAUNCH_IF_NEEDED = True


## 2. Setup


In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt

repo_search_start = Path.cwd().resolve()
REPO_ROOT = next(
    candidate
    for candidate in (repo_search_start, *repo_search_start.parents)
    if (candidate / "hardware_capture" / "rf" / "redstone" / "sequence.py").exists()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REDSTONE_DIR = REPO_ROOT / "hardware_capture" / "rf" / "redstone"
OUTPUT_DIR = REDSTONE_DIR / "examples" / "generated" / "nvnmr_adiabatic_inversion_train"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_TPS_PATH = REDSTONE_DIR / "20260630_NVNMR_AdiabaticInversionTrain.tps"
TEMPLATE_TNT_PATH = REDSTONE_DIR / "20260630_NVNMR_AdiabaticInversionTrain.tnt"
OUTPUT_STEM = "20260630_NVNMR_AdiabaticInversionTrain_chirp_waveforms"
OUTPUT_TPS_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps"
OUTPUT_TNT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tnt"
OUTPUT_JSON_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}.tps.json"
OUTPUT_P180_PHASE_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_phmod_p180_phase_table.txt"
OUTPUT_PN180_PHASE_TXT_PATH = OUTPUT_DIR / f"{OUTPUT_STEM}_phmod_pn180_phase_table.txt"

P180_PHASE_TABLE = "phmod_p180"
PN180_PHASE_TABLE = "phmod_pn180"
TARGET_PHASE_LINE = "F1_PhMod"
P180_PHASE_EVENTS = (5, 6)
PN180_PHASE_EVENTS = (12, 13)

from hardware_capture.rf.redstone import Chirp, Sequence, TNMRControl

assert TEMPLATE_TPS_PATH.exists(), TEMPLATE_TPS_PATH
assert TEMPLATE_TNT_PATH.exists(), TEMPLATE_TNT_PATH
assert OUTPUT_TPS_PATH.resolve() != TEMPLATE_TPS_PATH.resolve()
assert OUTPUT_TNT_PATH.resolve() != TEMPLATE_TNT_PATH.resolve()


## 3. Load And Inspect The Sequence


In [ ]:
sequence = Sequence.load(TEMPLATE_TPS_PATH)
parameter_values = sequence.parameter_values()
required_parameters = {"freq_neg", "freq_pos", "t_mod_offset", "t_p180", "t_pn180"}
missing_parameters = sorted(required_parameters - set(parameter_values))
if missing_parameters:
    raise ValueError(f"Missing required parameters: {missing_parameters}")

target_references = [
    reference
    for reference in sequence.table_references()
    if reference.table_name in {P180_PHASE_TABLE, PN180_PHASE_TABLE}
]

{
    "summary": sequence.summary().to_dict(),
    "parameters": parameter_values,
    "target_references": target_references,
}


## 4. Build Up And Down Chirps


In [ ]:
t_mod_offset_text = str(parameter_values["t_mod_offset"]).strip().lower()
if t_mod_offset_text.endswith("ns"):
    t_mod_offset_us = float(t_mod_offset_text[:-2]) * 1.0e-3
elif t_mod_offset_text.endswith("n"):
    t_mod_offset_us = float(t_mod_offset_text[:-1]) * 1.0e-3
elif t_mod_offset_text.endswith("us"):
    t_mod_offset_us = float(t_mod_offset_text[:-2])
elif t_mod_offset_text.endswith("u"):
    t_mod_offset_us = float(t_mod_offset_text[:-1])
else:
    raise ValueError(f"Expected t_mod_offset in ns or us, got {t_mod_offset_text!r}.")

half_sweep_hz = SWEEP_WIDTH_HZ / 2.0
freq_neg_hz = -half_sweep_hz
freq_pos_hz = half_sweep_hz
duration_us = POINTS * DWELL_US
duration_ms = duration_us * 1.0e-3
pulse_duration_us = duration_us - t_mod_offset_us
if pulse_duration_us <= 0:
    raise ValueError("Pulse duration must be positive after subtracting t_mod_offset.")

p180_waveform = Chirp(
    points=POINTS,
    sweep_width_hz=SWEEP_WIDTH_HZ,
    dwell_s=DWELL_US * 1.0e-6,
    offset_hz=-half_sweep_hz,
    phase=0.0,
    phase_steps=2**16,
)
pn180_waveform = Chirp(
    points=POINTS,
    sweep_width_hz=-SWEEP_WIDTH_HZ,
    dwell_s=DWELL_US * 1.0e-6,
    offset_hz=half_sweep_hz,
    phase=0.0,
    phase_steps=2**16,
)

p180_mod_freq = [value - p180_waveform.offset_hz for value in p180_waveform.frequency_values()]
pn180_mod_freq = [value - pn180_waveform.offset_hz for value in pn180_waveform.frequency_values()]

{
    "freq_neg_hz": freq_neg_hz,
    "freq_pos_hz": freq_pos_hz,
    "duration_us": duration_us,
    "t_mod_offset_us": t_mod_offset_us,
    "pulse_duration_us": pulse_duration_us,
    "p180_total_frequency_span_hz": (freq_neg_hz + p180_mod_freq[0], freq_neg_hz + p180_mod_freq[-1]),
    "pn180_total_frequency_span_hz": (freq_pos_hz + pn180_mod_freq[0], freq_pos_hz + pn180_mod_freq[-1]),
}


## 5. Visualize The Chirp Pulses


In [ ]:
time_ms = [index * p180_waveform.dwell_s * 1.0e3 for index in range(POINTS)]
p180_phase_deg = p180_waveform.phase_values()
pn180_phase_deg = pn180_waveform.phase_values()
if p180_phase_deg is None or pn180_phase_deg is None:
    raise ValueError("Phase output is not available.")

p180_total_freq = [freq_neg_hz + value for value in p180_mod_freq]
pn180_total_freq = [freq_pos_hz + value for value in pn180_mod_freq]
p180_i = [math.cos(math.radians(value)) for value in p180_phase_deg]
p180_q = [math.sin(math.radians(value)) for value in p180_phase_deg]
pn180_i = [math.cos(math.radians(value)) for value in pn180_phase_deg]
pn180_q = [math.sin(math.radians(value)) for value in pn180_phase_deg]

plot_step = max(1, POINTS // 5000)
plot_indices = list(range(0, POINTS, plot_step))
if plot_indices[-1] != POINTS - 1:
    plot_indices.append(POINTS - 1)

fig, axes = plt.subplots(4, 1, figsize=(10, 10), sharex=True)
axes[0].plot([time_ms[i] for i in plot_indices], [p180_mod_freq[i] * 1.0e-3 for i in plot_indices], label="p180 phase modulation")
axes[0].plot([time_ms[i] for i in plot_indices], [pn180_mod_freq[i] * 1.0e-3 for i in plot_indices], label="pn180 phase modulation")
axes[0].set_ylabel("Phase mod (kHz)")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="best")

axes[1].plot([time_ms[i] for i in plot_indices], [p180_total_freq[i] * 1.0e-3 for i in plot_indices], label="p180 total F1")
axes[1].plot([time_ms[i] for i in plot_indices], [pn180_total_freq[i] * 1.0e-3 for i in plot_indices], label="pn180 total F1")
axes[1].set_ylabel("Total F1 (kHz)")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="best")

axes[2].plot([time_ms[i] for i in plot_indices], [p180_phase_deg[i] for i in plot_indices], label="p180 phase")
axes[2].plot([time_ms[i] for i in plot_indices], [pn180_phase_deg[i] for i in plot_indices], label="pn180 phase")
axes[2].set_ylabel("Phase (deg)")
axes[2].set_ylim(-5, 365)
axes[2].grid(True, alpha=0.3)
axes[2].legend(loc="best")

axes[3].plot([time_ms[i] for i in plot_indices], [p180_i[i] for i in plot_indices], label="p180 I")
axes[3].plot([time_ms[i] for i in plot_indices], [p180_q[i] for i in plot_indices], label="p180 Q")
axes[3].plot([time_ms[i] for i in plot_indices], [pn180_i[i] for i in plot_indices], linestyle="--", label="pn180 I")
axes[3].plot([time_ms[i] for i in plot_indices], [pn180_q[i] for i in plot_indices], linestyle="--", label="pn180 Q")
axes[3].set_xlabel("Time (ms)")
axes[3].set_ylabel("Unit envelope")
axes[3].grid(True, alpha=0.3)
axes[3].legend(loc="best")

fig.suptitle(f"NVNMR adiabatic inversion chirps: {duration_ms:.3f} ms table")
fig.tight_layout()
plt.show()


## 6. Apply And Save The Sequence


In [ ]:
p180_rows = p180_waveform.phase_column(precision=3)
pn180_rows = pn180_waveform.phase_column(precision=3)
if p180_rows is None or pn180_rows is None:
    raise ValueError("Phase output is not available.")

sequence.set_waveform_to_table(P180_PHASE_TABLE, p180_waveform, quantity="phase", precision=3, phase_units="deg", allow_shared=True)
sequence.set_waveform_to_table(PN180_PHASE_TABLE, pn180_waveform, quantity="phase", precision=3, phase_units="deg", allow_shared=True)
for event_number in P180_PHASE_EVENTS:
    sequence.assign_table_to_event(TARGET_PHASE_LINE, event_number, P180_PHASE_TABLE)
for event_number in PN180_PHASE_EVENTS:
    sequence.assign_table_to_event(TARGET_PHASE_LINE, event_number, PN180_PHASE_TABLE)

sequence.set_parameter_value("freq_neg", f"{freq_neg_hz:.15g}")
sequence.set_parameter_value("freq_pos", f"{freq_pos_hz:.15g}")
sequence.set_parameter_value("t_p180", f"{pulse_duration_us:.15g}u")
sequence.set_parameter_value("t_pn180", f"{pulse_duration_us:.15g}u")

p180_waveform.write_phase_column(OUTPUT_P180_PHASE_TXT_PATH, precision=3)
pn180_waveform.write_phase_column(OUTPUT_PN180_PHASE_TXT_PATH, precision=3)
sequence.write_json(OUTPUT_JSON_PATH, include_raw_bytes=False)
sequence.write_tps(OUTPUT_TPS_PATH)

reloaded = Sequence.load(OUTPUT_TPS_PATH)
reloaded_parameters = reloaded.parameter_values()
reloaded_tables = {table.get("name"): table for table in reloaded.tables}
assert reloaded_parameters["freq_neg"] == f"{freq_neg_hz:.15g}"
assert reloaded_parameters["freq_pos"] == f"{freq_pos_hz:.15g}"
assert reloaded_parameters["t_p180"] == f"{pulse_duration_us:.15g}u"
assert reloaded_parameters["t_pn180"] == f"{pulse_duration_us:.15g}u"
assert len((reloaded_tables[P180_PHASE_TABLE].get("entries") or "").splitlines()) == POINTS
assert len((reloaded_tables[PN180_PHASE_TABLE].get("entries") or "").splitlines()) == POINTS

{
    "freq_neg": reloaded_parameters["freq_neg"],
    "freq_pos": reloaded_parameters["freq_pos"],
    "t_p180": reloaded_parameters["t_p180"],
    "t_pn180": reloaded_parameters["t_pn180"],
    "p180_rows": len(p180_rows),
    "pn180_rows": len(pn180_rows),
}


## 7. Open The TNT, Edit Parameters/Tables, And Save A Copy


In [ ]:
if not RUN_TNMR_SAVE_AS:
    raise RuntimeError("Set RUN_TNMR_SAVE_AS = True to create the output TNT through TNMR.")

control = TNMRControl()
control.connect(launch=TNMR_LAUNCH_IF_NEEDED)
try:
    control.open(TEMPLATE_TNT_PATH)
    open_fallback_error = None
except RuntimeError as exc:
    open_fallback_error = str(exc)
    control.bind_active_document()
    if Path(control.path).resolve() != TEMPLATE_TNT_PATH.resolve():
        raise RuntimeError(f"TNMR active document is not the source TNT: {control.path}") from exc

parameter_names = ("freq_neg", "freq_pos", "t_p180", "t_pn180", "t_mod_offset")
before_parameters = {name: str(control.get_parameter(name)) for name in parameter_names}
before_table_counts = {
    P180_PHASE_TABLE: len(str(control.get_table(P180_PHASE_TABLE)).split()),
    PN180_PHASE_TABLE: len(str(control.get_table(PN180_PHASE_TABLE)).split()),
}

control.load_sequence(OUTPUT_TPS_PATH)
control.set_parameter("freq_neg", f"{freq_neg_hz:.15g}")
control.set_parameter("freq_pos", f"{freq_pos_hz:.15g}")
control.set_parameter("t_p180", f"{pulse_duration_us:.15g}u")
control.set_parameter("t_pn180", f"{pulse_duration_us:.15g}u")
control.set_table(P180_PHASE_TABLE, p180_rows)
control.set_table(PN180_PHASE_TABLE, pn180_rows)

after_parameters = {name: str(control.get_parameter(name)) for name in parameter_names}
after_table_counts = {
    P180_PHASE_TABLE: len(str(control.get_table(P180_PHASE_TABLE)).split()),
    PN180_PHASE_TABLE: len(str(control.get_table(PN180_PHASE_TABLE)).split()),
}
control.save_as(OUTPUT_TNT_PATH)

assert after_parameters["freq_neg"] == f"{freq_neg_hz:.15g}"
assert after_parameters["freq_pos"] == f"{freq_pos_hz:.15g}"
assert after_parameters["t_p180"] == f"{pulse_duration_us:.15g}u"
assert after_parameters["t_pn180"] == f"{pulse_duration_us:.15g}u"
assert after_table_counts[P180_PHASE_TABLE] == POINTS
assert after_table_counts[PN180_PHASE_TABLE] == POINTS
assert OUTPUT_TNT_PATH.exists()

{
    "output_tnt": OUTPUT_TNT_PATH,
    "tnt_bytes": OUTPUT_TNT_PATH.stat().st_size,
    "before_parameters": before_parameters,
    "after_parameters": after_parameters,
    "before_table_counts": before_table_counts,
    "after_table_counts": after_table_counts,
    "open_fallback_error": open_fallback_error,
}


## 8. Output Summary


In [ ]:
generated_files = [
    OUTPUT_P180_PHASE_TXT_PATH,
    OUTPUT_PN180_PHASE_TXT_PATH,
    OUTPUT_JSON_PATH,
    OUTPUT_TPS_PATH,
    OUTPUT_TNT_PATH,
]

[
    {"file": path.name, "path": str(path), "bytes": path.stat().st_size}
    for path in generated_files
]
